# Recertification des documents KYC (ID cards / justificatifs de résidence)

## Contexte
- Documents scannés de mauvaise qualité, parfois **photocopies de photocopies**
- **Rotation variable** (0°, 90°, 180°, 270° + skew fin)
- **Multi-pays / multi-formats** (CNI, passeports, titres de séjour, justificatifs de domicile)

## Décision d'architecture

Aucun des modèles listés, utilisé **seul**, n'est optimal. Le meilleur résultat vient d'un **pipeline hybride** :

1. **Prétraitement OpenCV** : correction de rotation grossière (0/90/180/270) + correction de skew fin + débruitage/contraste. Indispensable en amont de n'importe quel modèle.
2. **PaddleOCR** (PP-OCRv4 + classificateur d'angle `cls`) : très robuste sur scans dégradés/photocopies, détecte et corrige la rotation ligne par ligne, rapide → sert de **première passe OCR brute** et de **filtre qualité**.
3. **Qwen2.5-VL-7B-Instruct** (déjà sur votre ModelHub) : extraction **structurée** (JSON) des champs KYC (nom, prénom, date de naissance, n° de document, adresse, date d'expiration, etc.), grâce à sa bonne généralisation multi-pays/multi-formats et sa compréhension de mise en page.
4. **InternVL (OpenGVLab)** : utilisé en **second avis** sur le même document → si désaccord significatif avec Qwen sur un champ critique (n° de document, nom, date), le document est **flaggé pour revue humaine** au lieu d'être recertifié automatiquement. C'est une exigence de robustesse en conformité KYC.
5. **SmolVLM** : optionnel, utile uniquement comme **filtre rapide de tri** (document lisible vs illisible) avant d'engager les modèles lourds, pour économiser du calcul sur de gros volumes.
6. **DeepSeek-VL / LiquidAI VLM / ByteDance Douyin VLM** : non recommandés comme moteur principal (moins performants ou non spécialisés sur l'OCR documentaire) — non utilisés dans ce pipeline, mais le code reste modulaire pour les brancher si besoin.

## Sortie
Un enregistrement structuré par document :
```json
{
  "fichier": "...",
  "type_document": "id_card" | "residence_permit",
  "pays_detecte": "...",
  "champs_extraits": {...},
  "confiance": 0.0-1.0,
  "accord_modeles": true/false,
  "statut": "certifie" | "a_revoir_manuellement"
}
```


## 1. Imports et configuration des chemins ModelHub

In [ ]:
import os
import io
import json
import cv2
import numpy as np
from PIL import Image
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

import torch

# --- Chemins des modèles sur le ModelHub (à adapter si besoin) ---
MODEL_PATHS = {
    "qwen_vlm": "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen2.5-VL-7B-instruct/main",
    "internvl": "/domino/edv/modelhub/ModelHub-model-huggingface-OpenGVLab/InternVL2_5-8B/main",
    "smolvlm": "/domino/edv/modelhub/ModelHub-model-huggingface-HuggingFaceTB/SmolVLM-Instruct/main",
    # Modèles listés mais non utilisés comme moteur principal (voir justification ci-dessus)
    # "deepseek_vlm": "/domino/edv/modelhub/.../deepseek-vl-7b-chat/main",
    # "liquid_vlm": "/domino/edv/modelhub/.../LFM-VL/main",
    # "douyin_content_vlm": "/domino/edv/modelhub/.../douyin-content-vlm/main",
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device utilisé : {DEVICE}")
for name, path in MODEL_PATHS.items():
    print(f"{name:12s} -> exists={os.path.isdir(path)}  ({path})")


## 2. Prétraitement d'image : rotation, skew, débruitage, contraste

C'est l'étape **la plus critique** de tout le pipeline : sur des photocopies bruitées et
des rotations arbitraires, aucun modèle (OCR ou VLM) ne donne de bons résultats sans
correction préalable.

Stratégie :
1. Détection de la rotation grossière (0/90/180/270°) via **Tesseract OSD** (rapide, fiable sur texte net) *ou* via le classificateur d'angle de PaddleOCR (plus robuste sur documents dégradés — utilisé ici en fallback).
2. Correction du skew fin (quelques degrés) par transformée de Hough sur les contours de texte.
3. Débruitage (`fastNlMeansDenoising`) + amélioration de contraste adaptative (CLAHE) — utile en particulier sur les photocopies grisées/écrasées.


In [ ]:
import math

def denoise_and_enhance(img_bgr: np.ndarray) -> np.ndarray:
    """Débruitage + amélioration de contraste adaptatif, utile sur photocopies."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    denoised = cv2.fastNlMeansDenoising(gray, h=10, templateWindowSize=7, searchWindowSize=21)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enhanced = clahe.apply(denoised)
    # Léger seuillage adaptatif pour renforcer le contraste texte/fond sur photocopies très pâles
    enhanced_bgr = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)
    return enhanced_bgr


def detect_coarse_rotation_osd(img_bgr: np.ndarray) -> int:
    """Détecte la rotation grossière (0/90/180/270) via Tesseract OSD.
    Retourne l'angle à appliquer pour corriger (dans le sens horaire inverse)."""
    try:
        import pytesseract
        osd = pytesseract.image_to_osd(img_bgr)
        angle = int([l for l in osd.split("\n") if "Rotate" in l][0].split(":")[1].strip())
        return angle
    except Exception as e:
        print(f"[WARN] OSD Tesseract indisponible ou échec ({e}), rotation grossière non détectée ici.")
        return 0


def rotate_image(img_bgr: np.ndarray, angle: int) -> np.ndarray:
    """Rotation de l'image d'un angle multiple de 90° (ou arbitraire) en conservant tout le contenu."""
    if angle == 0:
        return img_bgr
    (h, w) = img_bgr.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    cos = np.abs(M[0, 0]); sin = np.abs(M[0, 1])
    new_w = int((h * sin) + (w * cos))
    new_h = int((h * cos) + (w * sin))
    M[0, 2] += (new_w / 2) - center[0]
    M[1, 2] += (new_h / 2) - center[1]
    return cv2.warpAffine(img_bgr, M, (new_w, new_h), borderValue=(255, 255, 255))


def detect_and_correct_skew(img_bgr: np.ndarray, max_angle: float = 15.0) -> np.ndarray:
    """Corrige un skew fin (quelques degrés) via transformée de Hough sur les lignes de texte."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150, apertureSize=3)
    lines = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=100,
                             minLineLength=gray.shape[1] // 4, maxLineGap=20)
    if lines is None:
        return img_bgr
    angles = []
    for line in lines:
        x1, y1, x2, y2 = line[0]
        angle = math.degrees(math.atan2(y2 - y1, x2 - x1))
        if -max_angle <= angle <= max_angle:
            angles.append(angle)
    if not angles:
        return img_bgr
    median_angle = float(np.median(angles))
    if abs(median_angle) < 0.5:
        return img_bgr
    return rotate_image(img_bgr, median_angle)


def preprocess_document(image_path: str) -> np.ndarray:
    """Pipeline complet de prétraitement : rotation grossière -> skew fin -> débruitage/contraste."""
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Impossible de lire l'image : {image_path}")

    coarse_angle = detect_coarse_rotation_osd(img)
    img = rotate_image(img, coarse_angle)

    img = detect_and_correct_skew(img)
    img = denoise_and_enhance(img)
    return img


## 3. Passe OCR robuste avec PaddleOCR (première passe + filtre qualité)

PaddleOCR fournit en un seul appel : détection de zones de texte, classification d'angle
ligne par ligne (utile si la rotation résiduelle est locale, ex. cachet tourné), et
reconnaissance. On l'utilise ici comme **vérité brute** pour comparer/valider la sortie du VLM.


In [ ]:
from paddleocr import PaddleOCR

# use_angle_cls=True : corrige les rotations locales fines par ligne de texte
paddle_ocr = PaddleOCR(use_angle_cls=True, lang="latin", show_log=False)
# Pour des documents non-latins (arabe, cyrillique, etc.), instancier un second moteur :
# paddle_ocr_ar = PaddleOCR(use_angle_cls=True, lang="ar", show_log=False)


def run_paddle_ocr(img_bgr: np.ndarray) -> dict:
    """Retourne le texte brut détecté + un score de confiance moyen."""
    result = paddle_ocr.ocr(img_bgr, cls=True)
    lines, scores = [], []
    for page in result:
        if not page:
            continue
        for box, (text, score) in page:
            lines.append(text)
            scores.append(score)
    avg_conf = float(np.mean(scores)) if scores else 0.0
    return {
        "raw_text": "\n".join(lines),
        "n_lines": len(lines),
        "avg_confidence": avg_conf,
    }


## 4. Chargement des VLM (Qwen2.5-VL et InternVL) depuis le ModelHub local

Les deux modèles sont chargés depuis leurs chemins locaux (pas de téléchargement HuggingFace Hub).


In [ ]:
from transformers import AutoModelForCausalLM, AutoProcessor, AutoTokenizer

# --- Qwen2.5-VL ---
from transformers import Qwen2_5_VLForConditionalGeneration

qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_PATHS["qwen_vlm"],
    torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
    local_files_only=True,
)
qwen_processor = AutoProcessor.from_pretrained(MODEL_PATHS["qwen_vlm"], local_files_only=True)

# --- InternVL (OpenGVLab) : utilisé comme second avis / cross-check ---
internvl_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATHS["internvl"],
    torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
    local_files_only=True,
    trust_remote_code=True,
)
internvl_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATHS["internvl"], local_files_only=True, trust_remote_code=True
)

print("Modèles chargés.")


## 5. Prompt d'extraction structurée (schéma JSON adaptatif par type de document)

Le prompt demande explicitement un JSON, avec un schéma qui s'adapte au type de document
(carte d'identité vs justificatif de résidence), afin de généraliser sur les formats
variables selon les pays.


In [ ]:
ID_CARD_SCHEMA = """{
  "type_document": "id_card",
  "pays_emetteur": "",
  "nom": "",
  "prenom": "",
  "date_naissance": "JJ/MM/AAAA",
  "numero_document": "",
  "date_expiration": "JJ/MM/AAAA",
  "sexe": "",
  "nationalite": ""
}"""

RESIDENCE_SCHEMA = """{
  "type_document": "residence_permit",
  "pays_emetteur": "",
  "nom": "",
  "prenom": "",
  "adresse_complete": "",
  "date_emission": "JJ/MM/AAAA",
  "date_expiration": "JJ/MM/AAAA",
  "numero_document": ""
}"""

def build_extraction_prompt(doc_type_hint: str = "auto") -> str:
    schema = ID_CARD_SCHEMA if doc_type_hint == "id_card" else RESIDENCE_SCHEMA
    return f"""Tu es un expert en vérification de documents KYC.
Analyse ce document (carte d'identité ou justificatif de résidence), qui peut être
mal scanné, être une photocopie, ou légèrement de travers.
Réponds UNIQUEMENT avec un objet JSON valide respectant strictement ce schéma
(laisser une valeur vide "" si un champ est illisible ou absent) :

{schema}

Ne donne aucun texte en dehors du JSON."""


## 6. Fonctions d'inférence Qwen2.5-VL et InternVL

In [ ]:
def extract_with_qwen(img_pil: Image.Image, doc_type_hint: str = "auto") -> dict:
    prompt = build_extraction_prompt(doc_type_hint)
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img_pil},
                {"type": "text", "text": prompt},
            ],
        }
    ]
    text_prompt = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = qwen_processor(text=[text_prompt], images=[img_pil], return_tensors="pt").to(qwen_model.device)

    with torch.no_grad():
        output_ids = qwen_model.generate(**inputs, max_new_tokens=512, do_sample=False)
    output_text = qwen_processor.batch_decode(
        output_ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True
    )[0]

    return safe_json_parse(output_text)


def extract_with_internvl(img_pil: Image.Image, doc_type_hint: str = "auto") -> dict:
    prompt = build_extraction_prompt(doc_type_hint)
    # InternVL (trust_remote_code) expose généralement une méthode .chat(tokenizer, pixel_values, question, ...)
    pixel_values = internvl_model.load_image(img_pil) if hasattr(internvl_model, "load_image") else None
    if pixel_values is not None:
        pixel_values = pixel_values.to(internvl_model.device, dtype=internvl_model.dtype)
        response = internvl_model.chat(internvl_tokenizer, pixel_values, prompt, dict(max_new_tokens=512))
    else:
        # Fallback générique si l'API diffère selon la version du modèle
        response = "{}"
    return safe_json_parse(response)


def safe_json_parse(text: str) -> dict:
    """Extrait le premier bloc JSON valide d'une sortie de modèle (tolérant aux ``` et texte parasite)."""
    text = text.strip()
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1:
        return {"_parse_error": True, "_raw_output": text}
    try:
        return json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return {"_parse_error": True, "_raw_output": text}


## 7. Réconciliation des deux VLM + score de confiance

On compare les champs critiques (nom, numéro de document, dates). En cas de désaccord
sur un champ critique, le document est marqué `a_revoir_manuellement` plutôt que
`certifie` automatiquement — règle de prudence indispensable en contexte KYC.


In [ ]:
CRITICAL_FIELDS = ["nom", "prenom", "numero_document", "date_naissance", "date_expiration"]

def normalize(v: Optional[str]) -> str:
    if v is None:
        return ""
    return str(v).strip().lower().replace(" ", "")

def reconcile(qwen_out: dict, internvl_out: dict, paddle_ocr_result: dict) -> dict:
    disagreements = []
    for f in CRITICAL_FIELDS:
        v1 = normalize(qwen_out.get(f))
        v2 = normalize(internvl_out.get(f))
        if v1 and v2 and v1 != v2:
            disagreements.append(f)

    agree = len(disagreements) == 0
    # Score de confiance combinant : accord inter-modèles + confiance OCR brute
    base_conf = paddle_ocr_result.get("avg_confidence", 0.0)
    confidence = base_conf * (1.0 if agree else 0.5)

    statut = "certifie" if (agree and confidence >= 0.75) else "a_revoir_manuellement"

    return {
        "champs_extraits": qwen_out,  # Qwen comme source primaire si accord
        "champs_extraits_internvl": internvl_out,
        "champs_en_desaccord": disagreements,
        "confiance": round(confidence, 3),
        "accord_modeles": agree,
        "statut": statut,
    }


## 8. Pipeline complet : d'un fichier image à l'enregistrement de recertification

In [ ]:
def recertify_document(image_path: str, doc_type_hint: str = "auto") -> dict:
    """Pipeline complet : prétraitement -> OCR -> double VLM -> réconciliation."""
    corrected_bgr = preprocess_document(image_path)
    img_pil = Image.fromarray(cv2.cvtColor(corrected_bgr, cv2.COLOR_BGR2RGB))

    paddle_result = run_paddle_ocr(corrected_bgr)

    qwen_out = extract_with_qwen(img_pil, doc_type_hint)
    internvl_out = extract_with_internvl(img_pil, doc_type_hint)

    reconciliation = reconcile(qwen_out, internvl_out, paddle_result)

    record = {
        "fichier": os.path.basename(image_path),
        "type_document": qwen_out.get("type_document", doc_type_hint),
        "pays_detecte": qwen_out.get("pays_emetteur", ""),
        "ocr_brut_paddle": paddle_result["raw_text"],
        **reconciliation,
    }
    return record


## 9. Traitement par lot sur un dossier de documents

In [ ]:
import pandas as pd

def batch_recertify(folder_path: str, doc_type_hint: str = "auto") -> pd.DataFrame:
    folder = Path(folder_path)
    exts = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
    records = []
    for f in sorted(folder.iterdir()):
        if f.suffix.lower() not in exts:
            continue
        try:
            rec = recertify_document(str(f), doc_type_hint)
        except Exception as e:
            rec = {"fichier": f.name, "erreur": str(e), "statut": "erreur_traitement"}
        records.append(rec)
        print(f"{f.name:40s} -> statut = {rec.get('statut')}")
    return pd.DataFrame(records)

# Exemple d'utilisation :
# df_resultats = batch_recertify("/mnt/user-data/uploads/kyc_documents", doc_type_hint="auto")
# df_resultats.to_csv("/mnt/user-data/outputs/resultats_recertification.csv", index=False)
# df_resultats


## 10. Exemple sur un document unique

In [ ]:
# Exemple :
# result = recertify_document("/mnt/user-data/uploads/id_card_maroc_photocopie.jpg", doc_type_hint="id_card")
# print(json.dumps(result, indent=2, ensure_ascii=False))


## Notes de mise en production

- **Volumétrie** : pour de gros volumes, utiliser SmolVLM en pré-filtre (lisible/illisible) avant d'engager Qwen2.5-VL + InternVL, qui sont plus coûteux en calcul.
- **Langues non-latines** : instancier des instances PaddleOCR supplémentaires (`lang="ar"`, `lang="cyrillic"`, etc.) selon les pays couverts.
- **Traçabilité KYC** : conserver systématiquement l'image prétraitée + les deux sorties JSON brutes (Qwen et InternVL) pour audit, même en cas de certification automatique.
- **Amélioration continue** : les documents `a_revoir_manuellement` corrigés par un opérateur peuvent servir de jeu de données pour un fine-tuning léger (LoRA) de Qwen2.5-VL sur vos formats spécifiques.
